# Collections Recovery Analytics
**Purpose:** audit the source data, create a governed analytical layer, test the 11% MoM claim, and define a defensible investment decision.

> Raw data is immutable. This notebook documents the reasoning and the reproducible checks.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
RAW=Path('../data/raw')
files=sorted(RAW.glob('*.csv'))
print(f'{len(files)} CSV files found')


In [ ]:
audit=pd.read_csv('../evidence/data_audit.csv')
audit[['dataset','rows','columns','exact_duplicate_rows','missing_cells','unique_ids','duplicate_id_groups']].sort_values('dataset')

In [ ]:
qa=pd.read_csv('../evidence/data_quality_findings.csv')
qa.sort_values('count',ascending=False).head(20)

## Golden dataset rules
- Anchor borrower identity to `accounts.account_id -> accounts.borrower_id`.
- Remove exact duplicate payment/call rows programmatically.
- Quarantine conflicting call IDs.
- Flag reused payment references instead of assuming they are duplicates.
- Mark August as partial.
- Treat the recovery metric as a proxy because monthly balance history is unavailable.

In [ ]:
m=pd.read_csv('../data/golden/monthly_recovery_summary.csv')
m[['month','recovered_amount','proxy_recovery_rate','mom_recovery_change_pct','is_partial_month']]

In [ ]:
feb=0.016598026631322794; mar=0.01842241712141402; change=(mar/feb-1)*100
print(f'February proxy recovery: {feb:.3%}')
print(f'March proxy recovery: {mar:.3%}')
print(f'February→March MoM change: {change:.1f}%')
print('Conclusion: numerically close to the stated 11% claim, but not sufficient to establish causality.')

In [ ]:
seg=pd.read_csv('../data/golden/segment_performance.csv')
seg.sort_values('recovery_rate',ascending=False).head(15)

In [ ]:
strategy=pd.read_csv('../data/golden/strategy_performance_descriptive.csv')
strategy

## Investment conclusion
The data is best used to prioritize a controlled targeting experiment. Observational strategy/channel lift should not be converted directly into a ₹10 Cr ROI claim because campaign windows are inconsistent and targeting is not randomized.